# Test: Qwen3-32B — Validator V1

GPU 4 (shared with Mistral-Small, 45% utilization), Port 8002

**Prerequisites:** vLLM server running on port 8002

In [ ]:
import sys
sys.path.insert(0, '/storage/data/AgenticCyOps_Private')

from models.utils import Qwen3_32B

model = Qwen3_32B()
print('Model config:')
model.get_config()

## 1. Health Check

In [ ]:
assert model.health_check(), 'Server not running on port 8002!'
print('Health check passed')

## 2. List Models

In [ ]:
models = model.list_models()
for m in models:
    print(f'  {m.id}')

## 3. Chat Completions

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

# 3a. Basic chat
resp = model.chat(messages, max_tokens=100)
print('Basic chat:', resp.choices[0].message.content)

In [ ]:
# 3b. Deterministic
resp = model.chat_deterministic(messages, max_tokens=100)
print('Deterministic:', resp.choices[0].message.content)

In [ ]:
# 3c. Creative
resp = model.chat_creative(messages, max_tokens=100)
print('Creative:', resp.choices[0].message.content)

In [ ]:
# 3d. Streaming
stream = model.chat(messages, max_tokens=100, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

## 4. Validate (Primary Use Case)

As Validator V1, the primary function is `validate()` — judging agent proposals.

In [ ]:
# 4a. Safe proposal — should approve
resp = model.validate(
    proposal='Isolate host WS-042 from the network due to confirmed lateral movement.',
    context='Alert: Lateral movement detected from WS-042 to DC-01 via PsExec. Source IP 10.0.5.42, destination 10.0.1.10. MITRE T1570.'
)
print('Safe proposal:')
print(resp.choices[0].message.content)

In [ ]:
# 4b. Dangerous proposal — should reject or flag risks
resp = model.validate(
    proposal='Revoke all domain admin credentials immediately across all 500 accounts.',
    context='Alert: Single phishing email detected targeting user john.doe@corp.com. No evidence of credential compromise.'
)
print('Dangerous proposal:')
print(resp.choices[0].message.content)

In [ ]:
# 4c. Batch validate
proposals = [
    {
        'proposal': 'Block IP 10.0.5.12 at the firewall.',
        'context': 'Confirmed C2 communication from 10.0.5.12 to known malicious domain.'
    },
    {
        'proposal': 'Delete all firewall rules to reset network state.',
        'context': 'Minor configuration drift detected in firewall rule set.'
    },
    {
        'proposal': 'Quarantine malware sample in sandbox.',
        'context': 'EDR flagged suspicious executable on endpoint WS-015.'
    }
]
results = model.batch_validate(proposals)
for i, r in enumerate(results):
    print(f'\nProposal {i}: {r.choices[0].message.content[:120]}...')

## 5. Tool Calling

In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'query_siem',
            'description': 'Search SIEM logs',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string'},
                    'time_range': {'type': 'string'},
                },
                'required': ['query']
            }
        }
    }
]

# Note: Tool calling requires serve(enable_tool_choice=True)
# Default for validators is enable_tool_choice=False
# This test verifies it works when explicitly enabled
tc_messages = [
    {'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12'}
]
try:
    resp = model.tool_call(tc_messages, tools)
    tc = resp.choices[0].message.tool_calls
    if tc:
        print(f'Tool call: {tc[0].function.name}({tc[0].function.arguments})')
    else:
        print('No tool call made (expected if serve started without --enable-auto-tool-choice)')
except Exception as e:
    print(f'Tool calling not available (expected for default validator config): {e}')

## 6. Structured Output

In [ ]:
# JSON mode
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify: "Failed SSH from 10.0.5.12". Return {"severity": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.choices[0].message.content)

## 7. Batch Chat

In [ ]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.choices[0].message.content}')

## 8. Token Usage

In [ ]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

## 9. Get Config

In [ ]:
import json
print(json.dumps(model.get_config(), indent=2))

## Summary

All tests passed if no cells raised exceptions above.